# Exercise 12 - Feature Enginering

this is my work for exercise 12. i am using the radiation scan data from lab 12 and computing 3 extra features from the list.

The 3 features i picked:
1. Hotspot flag
2. Sensor Variability Index
3. ZScore vs Global Mean

## Step 1 - Making the CONVERT_XY function

First we need a UDF that takes easting and northing coordiantes and figures out which tile, survey unit, and subcell each point belongs to. the python version takes all the grid paramaters, and then we make a simpler SQL wrapper below that has the defualt values already filled in.

In [ ]:
%%sql -r udf_python_result
USE DATABASE "USER$FOX";
USE SCHEMA PUBLIC;
USE WAREHOUSE SNOWFLAKE_LEARNING_WH;

CREATE OR REPLACE FUNCTION DATA5035.FOX.CONVERT_XY(
    X FLOAT, Y FLOAT,
    ORIGIN_X FLOAT, ORIGIN_Y FLOAT,
    SU_SIZE FLOAT,
    TILE_GRID_X NUMBER, TILE_GRID_Y NUMBER,
    SUBCELL_GRID NUMBER
)
RETURNS OBJECT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
HANDLER = 'convert_xy'
AS
$$
def convert_xy(x, y, origin_x, origin_y, su_size, tile_grid_x, tile_grid_y, subcell_grid):
    dx = x - origin_x
    dy = y - origin_y
    tile_width  = su_size * tile_grid_x
    tile_height = su_size * tile_grid_y
    tile_col_idx = int(dx / tile_width)
    tile_row_idx = int(dy / tile_height)
    tile_label = chr(ord('A') + tile_row_idx) + chr(ord('A') + tile_col_idx)
    tile_dx = dx - tile_col_idx * tile_width
    tile_dy = dy - tile_row_idx * tile_height
    su_col = int(tile_dx / su_size)
    su_row = int(tile_dy / su_size)
    su_row_from_top = (tile_grid_y - 1) - su_row
    su_number = su_row_from_top * tile_grid_x + su_col + 1
    subcell_size = su_size / subcell_grid
    su_dx = tile_dx - su_col * su_size
    su_dy = tile_dy - su_row * su_size
    sc_col = int(su_dx / subcell_size)
    sc_row = int(su_dy / subcell_size)
    subcell_number = sc_row * subcell_grid + sc_col + 1
    return {"tile": tile_label, "su": su_number, "subcell": subcell_number}
$$

In [ ]:
%%sql -r udf_sql_result
CREATE OR REPLACE FUNCTION DATA5035.FOX.CONVERT_XY(X FLOAT, Y FLOAT)
RETURNS OBJECT
LANGUAGE SQL
AS '
    SELECT DATA5035.FOX.CONVERT_XY(
        X, Y,
        2180160.0001,
        6660000.0000,
        32.81,
        21,
        18,
        10
    )
'

## Step 2 - Base view with grid info

Now, create a view that adds tile, SU, and subcell colums to every row. this way i dont have to call the function every time, i can just query this view for the rest of the features.

In [ ]:
%%sql -r base_view_result
CREATE OR REPLACE VIEW DATA5035.FOX.V_SCAN_BASE AS
SELECT
    EASTING,
    NORTHING,
    READING,
    CONVERT_XY(EASTING, NORTHING) AS COORDS,
    COORDS:tile::STRING    AS TILE,
    COORDS:su::INTEGER     AS SU,
    COORDS:subcell::INTEGER AS SUBCELL
FROM DATA5035.SPRING26.SDG_001_RA226_SCANDATA

In [ ]:
%%sql -r base_preview
SELECT * FROM DATA5035.FOX.V_SCAN_BASE LIMIT 10

## Feature 1 - Hotspot Flag (Simplify)

this one checks if the max reading in a tile is above 7.4. i picked 7.4 becuase thats around the 99th percentile of all the readings. if its above that, the tile gets flagged as 1 (hot), otherwise 0.

In [ ]:
%%sql -r hotspot_result
CREATE OR REPLACE VIEW DATA5035.FOX.V_FEATURE_HOTSPOT_FLAG AS
SELECT
    TILE,
    COUNT(*)           AS SAMPLE_COUNT,
    MAX(READING)       AS MAX_READING,
    CASE WHEN MAX(READING) > 7.4 THEN 1 ELSE 0 END AS HOTSPOT_FLAG
FROM DATA5035.FOX.V_SCAN_BASE
GROUP BY TILE
ORDER BY HOTSPOT_FLAG DESC, MAX_READING DESC

In [ ]:
%%sql -r hotspot_preview
SELECT * FROM DATA5035.FOX.V_FEATURE_HOTSPOT_FLAG

## Feature 2 - Sensor Variability index 

this calcualtes stddev / avg for each tile. its basically the coefficient of variation. if a tile has a high number here it means the readings are all over the place, not consistant

In [ ]:
%%sql -r variability_result
CREATE OR REPLACE VIEW DATA5035.FOX.V_FEATURE_VARIABILITY_INDEX AS
SELECT
    TILE,
    COUNT(*)             AS SAMPLE_COUNT,
    ROUND(AVG(READING), 5)    AS AVG_READING,
    ROUND(STDDEV(READING), 5) AS STDDEV_READING,
    ROUND(STDDEV(READING) / NULLIF(AVG(READING), 0), 5) AS VARIABILITY_INDEX
FROM DATA5035.FOX.V_SCAN_BASE
GROUP BY TILE
ORDER BY VARIABILITY_INDEX DESC

In [ ]:
%%sql -r variability_preview
SELECT * FROM DATA5035.FOX.V_FEATURE_VARIABILITY_INDEX

## Feature 3 - ZScore vs Global Mean (Assess)

This one compares each tiles average to the overall average of the whole dataset. The formula is (tile_avg - global_avg) / global_stddev. positive zscore mean the tile is hotter then average, negative means its cooler

In [ ]:
%%sql -r zscore_result
CREATE OR REPLACE VIEW DATA5035.FOX.V_FEATURE_ZSCORE AS
WITH GLOBAL_STATS AS (
    SELECT
        AVG(READING)    AS GLOBAL_AVG,
        STDDEV(READING) AS GLOBAL_STDDEV
    FROM DATA5035.FOX.V_SCAN_BASE
),
TILE_STATS AS (
    SELECT
        TILE,
        COUNT(*)       AS SAMPLE_COUNT,
        AVG(READING)   AS TILE_AVG
    FROM DATA5035.FOX.V_SCAN_BASE
    GROUP BY TILE
)
SELECT
    t.TILE,
    t.SAMPLE_COUNT,
    ROUND(t.TILE_AVG, 5)                                         AS TILE_AVG,
    ROUND(g.GLOBAL_AVG, 5)                                       AS GLOBAL_AVG,
    ROUND(g.GLOBAL_STDDEV, 5)                                    AS GLOBAL_STDDEV,
    ROUND((t.TILE_AVG - g.GLOBAL_AVG) / g.GLOBAL_STDDEV, 5)     AS ZSCORE
FROM TILE_STATS t
CROSS JOIN GLOBAL_STATS g
ORDER BY ZSCORE DESC

In [ ]:
%%sql -r zscore_preview
SELECT * FROM DATA5035.FOX.V_FEATURE_ZSCORE

## Summary

So basically I made 3 features:
- **Hotspot Flag** - finds the tiles with really high readings
- **Variability Index** - shows which tiles have unstable readings
- **Z-Score** - compares each tile to the overal average

Everything is saved as views in DATA5035.SPRING26.SDG_001_RA226_SCANDATA